# Notebook for validating higher-order multipoles within the `meer21cm` pipeline 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from specs import *
from meer21cm import MockSimulation
from func_multipole import *
from meer21cm.telescope import dish_beam_sigma
from matplotlib.colors import LogNorm
from astropy.cosmology import Planck18
import matplotlib.ticker as tck
import os

In [ ]:
# Create output directory
os.makedirs('fullsim_multipole_plots', exist_ok=True)

# Load k-modes
karr = get_k_modes()
kperp, kpara, kmode, kvec = karr

In [ ]:
# Load multipole data
data = np.load('/users/dtassie/foregroundsims/meer21cm/papers/validation/00_multipole.npz')

In [ ]:
# Extract multipole moments for HI (tracer 1)
pdata_ell0 = data['pdata_ell0_arr']
pdata_ell2 = data['pdata_ell2_arr']
pdata_ell4 = data['pdata_ell4_arr']

phimod_ell0 = data['phimod_ell0_arr']
phimod_ell2 = data['phimod_ell2_arr']
phimod_ell4 = data['phimod_ell4_arr']

# Extract multipole moments for galaxies (tracer 2)
pg_ell0 = data['pg_ell0_arr']
pg_ell2 = data['pg_ell2_arr']
pg_ell4 = data['pg_ell4_arr']

pgmod_ell0 = data['pgmod_ell0_arr']
pgmod_ell2 = data['pgmod_ell2_arr']
pgmod_ell4 = data['pgmod_ell4_arr']

# Extract cross multipole moments
pcross_ell0 = data['pcross_ell0_arr']
pcross_ell2 = data['pcross_ell2_arr']
pcross_ell4 = data['pcross_ell4_arr']

pcrossmod_ell0 = data['pcrossmod_ell0_arr']
pcrossmod_ell2 = data['pcrossmod_ell2_arr']
pcrossmod_ell4 = data['pcrossmod_ell4_arr']

In [ ]:
# Extract 2D cylindrical power spectra
pdata_cy = data['pdata_cy_arr']
phimod_cy = data['phimod_cy_arr']
pg_cy = data['pg_cy_arr']
pgmod_cy = data['pgmod_cy_arr']
pcross_cy = data['pcross_cy_arr']
pcrossmod_cy = data['pcrossmod_cy_arr']

keff = data['keff']

In [ ]:
plt.rcParams.update({
    'font.size': 22,
    'axes.labelsize': 32,
    'axes.titlesize': 36,
    'xtick.labelsize': 26,
    'ytick.labelsize': 26,
    'legend.fontsize': 24,
})

ratio_min, ratio_max = (-1.6, 1.6)

# Prepare data for all three multipoles
multipoles = [0, 2, 4]
pdata_ells = [pdata_ell0, pdata_ell2, pdata_ell4]
phimod_ells = [phimod_ell0, phimod_ell2, phimod_ell4]
pg_ells = [pg_ell0, pg_ell2, pg_ell4]
pgmod_ells = [pgmod_ell0, pgmod_ell2, pgmod_ell4]
pcross_ells = [pcross_ell0, pcross_ell2, pcross_ell4]
pcrossmod_ells = [pcrossmod_ell0, pcrossmod_ell2, pcrossmod_ell4]

# Unit conversions
keff_h = keff / Planck18.h
hi_plot_scale = 1e6 * Planck18.h**3
gal_plot_scale = Planck18.h**3 / 1e3
cross_plot_scale = 1e3 * Planck18.h**3

In [ ]:
# Create separate figures for each multipole with HI, galaxy, cross as 3 panels on same row
for row, (ell, pdata_ell, phimod_ell, pg_ell, pgmod_ell, pcross_ell, pcrossmod_ell) in enumerate(
    zip(multipoles, pdata_ells, phimod_ells, pg_ells, pgmod_ells, pcross_ells, pcrossmod_ells)
):
    fig, axes = plt.subplots(
        3, 3, figsize=(50, 12), sharex=True,
        height_ratios=[2, 1, 1], dpi=150,
        gridspec_kw={'wspace': 0.1},
    )
    for ax in axes.ravel():
        ax.tick_params(axis='both', which='major', labelsize=26)
    
    # HI power (first column)
    pmockarr_hi = pdata_ell * hi_plot_scale
    pmodelarr_hi = phimod_ell * hi_plot_scale
    n_mocks = len(pmockarr_hi)
    
    axes[0, 0].errorbar(
        keff_h,
        pmockarr_hi.mean(axis=0) * keff_h,
        yerr=pmockarr_hi.std(axis=0) * keff_h / np.sqrt(10),
        label="Mock",
        ls='None',
        elinewidth=4,
        marker='s',
        markersize=8,
    )
    axes[0, 0].plot(keff_h, pmodelarr_hi.mean(axis=0) * keff_h, label="model", ls="--", color='C1', lw=3)
    axes[0, 0].set_ylim((pmodelarr_hi.mean(axis=0) * keff_h).min() * 0.8, (pmodelarr_hi.mean(axis=0) * keff_h).max() * 1.2)
    axes[0, 0].legend()
    axes[0, 0].set_title(r'$k P_{{\mathrm{{HI}}}} (k) \, [\mathrm{{mK}}^2\, \mathrm{{Mpc}}^2\, h^{{-2}}]$ ($\ell={}$)'.format(ell))
    
    axes[1, 0].plot(
        keff_h,
        (pmockarr_hi.mean(axis=0) - pmodelarr_hi.mean(axis=0)) / (pmockarr_hi.std(axis=0)) * np.sqrt(10),
        color='firebrick',
        lw=3,
    )
    axes[1, 0].fill_between(
        np.linspace(keff_h.min() - 0.005, keff_h.max() + 0.005, 100),
        -1,
        1,
        color="black",
        alpha=0.2,
    )
    axes[1, 0].axhline(0, color="black", ls="--", lw=3)
    axes[1, 0].set_xlim(keff_h.min() - 0.005, keff_h.max() + 0.005)
    axes[1, 0].set_ylim(ratio_min, ratio_max)
    axes[1, 0].set_ylabel(r'$\Delta P \, / \, \sigma_P$', labelpad=20)
    
    axes[2, 0].plot(
        keff_h,
        (pmockarr_hi.mean(axis=0)) / (pmodelarr_hi.mean(axis=0)) - 1,
        lw=3,
        color='C4',
    )
    axes[2, 0].axhline(0, color="black", ls="--")
    axes[2, 0].fill_between(
        np.linspace(keff_h.min() - 0.005, keff_h.max() + 0.005, 100),
        -0.05,
        0.05,
        color="black",
        alpha=0.2,
    )
    axes[2, 0].set_xlim(keff_h.min() - 0.005, keff_h.max() + 0.005)
    axes[2, 0].set_ylim(-0.15, 0.15)
    axes[2, 0].set_xlabel(r'$k\,[h{\rm Mpc^{-1}}]$', fontsize=40)
    axes[2, 0].set_ylabel(r'$\Delta P \, / \,P$', labelpad=-1, fontsize=32)
    
    # Galaxy power (middle column)
    pmockarr_gal = pg_ell * gal_plot_scale
    pmodelarr_gal = pgmod_ell * gal_plot_scale
    
    axes[0, 1].errorbar(
        keff_h,
        pmockarr_gal.mean(axis=0) * keff_h,
        yerr=pmockarr_gal.std(axis=0) * keff_h / np.sqrt(10),
        label="Mock",
        ls='None',
        elinewidth=4,
        marker='s',
        markersize=8,
    )
    axes[0, 1].plot(keff_h, pmodelarr_gal.mean(axis=0) * keff_h, label="model", ls="--", color='C1', lw=3)
    axes[0, 1].set_ylim((pmodelarr_gal.mean(axis=0) * keff_h).min() * 0.8, (pmodelarr_gal.mean(axis=0) * keff_h).max() * 1.2)
    axes[0, 1].legend()
    axes[0, 1].set_title(r'$k P_{{\mathrm{{g}}}} (k) \, [\mathrm{{Mpc}}^2\, h^{{-2}}]$ ($\ell={}$)'.format(ell))
    
    axes[1, 1].plot(
        keff_h,
        (pmockarr_gal.mean(axis=0) - pmodelarr_gal.mean(axis=0)) / (pmockarr_gal.std(axis=0)) * np.sqrt(10),
        color='firebrick',
        lw=3,
    )
    axes[1, 1].fill_between(
        np.linspace(keff_h.min() - 0.005, keff_h.max() + 0.005, 100),
        -1,
        1,
        color="black",
        alpha=0.2,
    )
    axes[1, 1].axhline(0, color="black", ls="--", lw=3)
    axes[1, 1].set_xlim(keff_h.min() - 0.005, keff_h.max() + 0.005)
    axes[1, 1].set_ylim(ratio_min, ratio_max)
    axes[1, 1].set_yticklabels([])
    
    axes[2, 1].plot(
        keff_h,
        (pmockarr_gal.mean(axis=0)) / (pmodelarr_gal.mean(axis=0)) - 1,
        lw=3,
        color='C4',
    )
    axes[2, 1].axhline(0, color="black", ls="--")
    axes[2, 1].fill_between(
        np.linspace(keff_h.min() - 0.005, keff_h.max() + 0.005, 100),
        -0.05,
        0.05,
        color="black",
        alpha=0.2,
    )
    axes[2, 1].set_xlim(keff_h.min() - 0.005, keff_h.max() + 0.005)
    axes[2, 1].set_ylim(-0.15, 0.15)
    axes[2, 1].set_xlabel(r'$k\,[h{\rm Mpc^{-1}}]$', fontsize=40)
    axes[2, 1].set_yticklabels([])
    
    # Cross power (right column)
    pmockarr_cross = pcross_ell * cross_plot_scale
    pmodelarr_cross = pcrossmod_ell * cross_plot_scale
    
    axes[0, 2].errorbar(
        keff_h,
        pmockarr_cross.mean(axis=0) * keff_h,
        yerr=pmockarr_cross.std(axis=0) * keff_h / np.sqrt(10),
        label="Mock",
        ls='None',
        elinewidth=4,
        marker='s',
        markersize=8,
    )
    axes[0, 2].plot(keff_h, pmodelarr_cross.mean(axis=0) * keff_h, label="model", ls="--", color='C1', lw=3)
    axes[0, 2].set_ylim((pmodelarr_cross.mean(axis=0) * keff_h).min() * 0.8, (pmodelarr_cross.mean(axis=0) * keff_h).max() * 1.2)
    axes[0, 2].legend()
    axes[0, 2].set_title(r'$k P_{{\times}} (k) \, [\mathrm{{mK}}\, \mathrm{{Mpc}}^2\, h^{{-2}}]$ ($\ell = {}$)'.format(ell))
    
    axes[1, 2].plot(
        keff_h,
        (pmockarr_cross.mean(axis=0) - pmodelarr_cross.mean(axis=0)) / (pmockarr_cross.std(axis=0)) * np.sqrt(10),
        color='firebrick',
        lw=3,
    )
    axes[1, 2].fill_between(
        np.linspace(keff_h.min() - 0.005, keff_h.max() + 0.005, 100),
        -1,
        1,
        color="black",
        alpha=0.2,
    )
    axes[1, 2].axhline(0, color="black", ls="--", lw=3)
    axes[1, 2].set_xlim(keff_h.min() - 0.005, keff_h.max() + 0.005)
    axes[1, 2].set_ylim(ratio_min, ratio_max)
    axes[1, 2].set_yticklabels([])
    
    axes[2, 2].plot(
        keff_h,
        (pmockarr_cross.mean(axis=0)) / (pmodelarr_cross.mean(axis=0)) - 1,
        lw=3,
        color='C4',
    )
    axes[2, 2].axhline(0, color="black", ls="--")
    axes[2, 2].fill_between(
        np.linspace(keff_h.min() - 0.005, keff_h.max() + 0.005, 100),
        -0.05,
        0.05,
        color="black",
        alpha=0.2,
    )
    axes[2, 2].set_xlim(keff_h.min() - 0.005, keff_h.max() + 0.005)
    axes[2, 2].set_ylim(-0.15, 0.15)
    axes[2, 2].set_xlabel(r'$k\,[h{\rm Mpc^{-1}}]$', fontsize=40)
    axes[2, 2].set_yticklabels([])
    
    plt.savefig(f'fullsim_multipole_plots/fullsim_1d_ell{ell}.pdf', dpi=150, bbox_inches='tight')
    plt.close()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(40, 28), gridspec_kw={'wspace': 0.3, 'hspace': 0.3}, dpi=150)
for ax in axes.ravel():
    ax.tick_params(axis='both', which='major', labelsize=26)

xbins = kperpbins / Planck18.h
ybins = kparabins / Planck18.h

# Define tracers and their data
tracers = [
    ('HI', pdata_cy.mean(axis=0), phimod_cy[0], 0, 1e6 * Planck18.h**3, r'$P_{\rm HI}\,[ h^{-3} {\rm mK^2 Mpc^3}]$'),
    ('Galaxy', pg_cy.mean(axis=0), pgmod_cy[0], 1, Planck18.h**3 / 1e3, r'$P_{\rm g}\,[ 10^{-3} h^{-3}{\rm Mpc^3}]$'),
    ('Cross', pcross_cy.mean(axis=0), pcrossmod_cy[0], 2, 1e3 * Planck18.h**3, r'$P_{\rm x}\,[h^{-3} {\rm mK Mpc^3}]$'),
]

for tracer_name, pdata_cy_val, pmod_cy_val, col_idx, scale, ylabel in tracers:
    pdata_cy_scaled = pdata_cy_val * scale
    pmod_cy_scaled = pmod_cy_val * scale
    
    arr = np.array([pdata_cy_scaled.T, pmod_cy_scaled.T])
    vmin = np.nanmin(arr)
    vmax = np.nanmax(arr)
    
    # Mock data
    axes[0, col_idx].pcolormesh(
        xbins, ybins, pdata_cy_scaled.T,
        norm=LogNorm(vmin=vmin, vmax=vmax)
    )
    axes[0, col_idx].set_ylabel(r'$k_\parallel\,[h{\rm Mpc^{-1}}]$', fontsize=32)
    axes[0, col_idx].set_title(f'{tracer_name} Mock', fontsize=34)
    axes[0, col_idx].yaxis.set_minor_locator(tck.AutoMinorLocator())
    axes[0, col_idx].xaxis.set_minor_locator(tck.AutoMinorLocator())
    
    # Model
    im = axes[1, col_idx].pcolormesh(
        xbins, ybins, pmod_cy_scaled.T,
        norm=LogNorm(vmin=vmin, vmax=vmax)
    )
    axes[1, col_idx].set_ylabel(r'$k_\parallel\,[h{\rm Mpc^{-1}}]$', fontsize=32)
    axes[1, col_idx].set_title(f'{tracer_name} Model', fontsize=34)
    cbar = plt.colorbar(im, ax=axes[0:2, col_idx], location="right", fraction=0.046, pad=0.04)
    cbar.set_label(ylabel, labelpad=12, fontsize=28)
    cbar.ax.tick_params(labelsize=24)
    axes[1, col_idx].yaxis.set_minor_locator(tck.AutoMinorLocator())
    axes[1, col_idx].xaxis.set_minor_locator(tck.AutoMinorLocator())
    
    # Ratio
    im = axes[2, col_idx].pcolormesh(
        xbins, ybins, pdata_cy_scaled.T / pmod_cy_scaled.T,
        vmin=0.8, vmax=1.2, cmap="bwr"
    )
    axes[2, col_idx].set_ylabel(r'$k_\parallel\,[h{\rm Mpc^{-1}}]$', fontsize=32)
    axes[2, col_idx].set_xlabel(r'$k_\perp\,[h{\rm Mpc^{-1}}]$', fontsize=32)
    axes[2, col_idx].set_title(f'{tracer_name} Ratio', fontsize=34)
    cbar = plt.colorbar(im, ax=axes[2, col_idx], location="right", fraction=0.046, pad=0.04)
    cbar.set_label(r'Mock/Model', labelpad=12, fontsize=28)
    cbar.ax.tick_params(labelsize=24)
    axes[2, col_idx].yaxis.set_minor_locator(tck.AutoMinorLocator())
    axes[2, col_idx].xaxis.set_minor_locator(tck.AutoMinorLocator())

plt.savefig('fullsim_multipole_plots/fullsim_2d_cylindrical.pdf', dpi=150, bbox_inches='tight')
plt.close()

print("Plots saved to fullsim_multipole_plots/")